<a href="https://colab.research.google.com/github/jonhrh1348/aviation-delay/blob/feature%2Fapi_data_setup/00_Initial_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install virtualenv
# !virtualenv venv

In [ ]:
# Install packages if not installed

!pip install pyspark
!pip install kafka-python

# kafka-python>=2.3.0
# pyspark>=4.0.0


In [ ]:
# !pip install -r requirements.txt
# !pip freeze
# !pip list

In [ ]:
import datetime
import json
import requests
import time
from google.colab import userdata
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# 1. Start the builder or continue previous one
spark = SparkSession.builder \
    .appName("PySpark-InitialSetup") \
    .getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
# A quick sanity check
# print("Spark Session created successfully!")

In [ ]:
# 2. Get aviation data from Aviation Edge API (to handle batch, db storage and multiple cities later)
ae_api_key = userdata.get('AE_API_KEY')
code = 'SIN'
flight_types = ['arrival', 'departure']
all_flight_data = []

# Define date range
start_date = datetime.datetime(2025, 9, 1)
end_date = datetime.datetime(2026, 3, 19)
interval = datetime.timedelta(days=7)

aviation_api_url = "https://aviation-edge.com/v2/public/flightsHistory"

print(f"Starting data collection from {start_date.date()} to {end_date.date()}.")

# 3. Batch call APIs and store them
for f_type in flight_types:
    current_date = start_date
    while current_date < end_date:
        date_end = min(current_date + interval, end_date)

        date_from = current_date.strftime("%Y-%m-%d")
        date_to = date_end.strftime("%Y-%m-%d")

        aviation_request_url = (
            f'{aviation_api_url}?key={ae_api_key}&code={code}'
            f'&type={f_type}&date_from={date_from}&date_to={date_to}'
        )

        try:
            print(f"Fetching {f_type} for date_from: {date_from} to date_end: {date_end}")
            response = requests.get(aviation_request_url)
            data = response.json()

            if isinstance(data, list):
                all_flight_data.extend(data)
            else:
                all_flight_data.append(data)

        except Exception as e:
            print(f"An error occurred: {e}")

        current_date = date_end
        time.sleep(10) # delay to avoid hitting rate limits

print(f"Data collection complete. Total records batches collected: {len(all_flight_data)}")
# print(json.dumps(all_flight_data, indent=2)) # set to print a few instead

In [ ]:
flight_list = json.dumps(all_flight_data, indent=2)

# 4. Parallelize the JSON string into a flight collection RDD
flight_collection = spark.sparkContext.parallelize([flight_list])

# 5. Create a DataFrame for analysis
aviation_df = (spark.read.json(flight_collection)
    .select(
        "flight.iataNumber",
        "airline.name",
        "arrival.scheduledTime",
        "arrival.actualTime",
        "arrival.delay",
        "arrival.baggage",
        "departure.scheduledTime",
        "departure.actualTime",
        "departure.delay",
        "codeshared.airline.name",
        "codeshared.flight.iataNumber",)
    .toDF(
        "IATA_number",
        "Airline",
        "Arr_scheduled_time",
        "Arr_actual_time",
        "Arr_delay (mins)",
        "Arr_baggage (mins)",
        "Dep_scheduled_time",
        "Dep_actual_time",
        "Dep_delay (mins)",
        "Codeshared_airline",
        "Codeshared_flight_number")
)

# df.printSchema()
aviation_df.show(n=3, truncate=False)

In [ ]:
#
# New set of calls
# 1. Get historical weather data from Open Weather API (to handle batch, db storage and multiple cities later)

lat = 1.290
long = 103.852
ttl_cnt = 0
units_used = 'metric'
weather_api_key = userdata.get('WEATHER_API_KEY')
weather_api_url = "https://history.openweathermap.org/data/2.5/history/city"

# Define the start and end dates
current_date = datetime.datetime(2026, 2, 1)
end_date = datetime.datetime(2026, 3, 1)
interval = datetime.timedelta(days=7)
all_weather_data = []

# if end:
#     request_url = f'{request_url}&end={end}'

print(f"Iterating through Feb 2026 and collecting data:\n")

# 2. Batch call API for weather data
while current_date < end_date:
    segment_end = min(current_date + interval, end_date)
    start_epoch = int(current_date.timestamp())
    end_epoch = int(segment_end.timestamp())

    # Construct the URL for this specific window
    weather_request_url = (
        f'{weather_api_url}?lat={lat}&lon={long}'
        f'&type=hour&start={start_epoch}&end={end_epoch}'
        f'&appid={weather_api_key}&units={units_used}'
    )
    if ttl_cnt > 0:
      weather_request_url = f'{weather_request_url}&cnt={ttl_cnt}'

    try:
      print(f"Fetching Period: {current_date.date()} to {segment_end.date()}")
      weather_response = requests.get(weather_request_url)

      resp_data = weather_response.json()
      data = resp_data['list']  # get the city_id and cod

      if isinstance(data, list):
        all_weather_data.extend(data)
      else:
        all_weather_data.append(data)
    except Exception as e:
      print(f"An error occurred: {e}")

    current_date = segment_end

print(f"Data collection complete. Total records batches collected: {len(all_weather_data)}")
# print(json.dumps(all_weather_data, indent=2)) # set to print a few instead

In [ ]:
weather_list = json.dumps(all_weather_data, indent=2)

# 3. Parallelize the JSON string into a weather collection RDD
weather_collection = spark.sparkContext.parallelize([weather_list])

# 4. Create an initial DataFrame to inspect schema
raw_weather_df = spark.read.json(weather_collection)
schema_fields = raw_weather_df.columns
raw_weather_df = raw_weather_df.withColumns(
  {"weather": F.array_join(F.col("weather.main"), ", "),
   "weather_desc": F.array_join(F.col("weather.description"), ", "),
   "dt": F.from_unixtime(F.col("dt")),
  }
)

# raw_weather_df.show(n=3, truncate=False)
# raw_weather_df.printSchema()

# 5. Define selection expressions with safety checks for 'rain' and 'snow'
select_exprs = [
    F.col("dt").alias("date_obs"),
    F.col("main.temp").alias("Current_Temp  (\N{DEGREE SIGN}C)"),
    F.col("main.feels_like").alias("Feels_Like  (\N{DEGREE SIGN}C)"),
    F.col("main.pressure").alias("Pressure (hPa)"),
    F.col("main.humidity").alias("Humidity (%)"),
    F.col("main.temp_min").alias("Min_Temp (\N{DEGREE SIGN}C)"),
    F.col("main.temp_max").alias("Max_Temp (\N{DEGREE SIGN}C)"),
    F.col("wind.speed").alias("Wind_Speed (m/s)"),
    F.col("wind.deg").alias("Wind_Deg"),
    F.col("clouds.all").alias("Cloudiness (%)"),
    # Check for rain
    F.col("rain.1h") if "rain" in schema_fields else F.lit(None).alias("Rainfall (mm)"),
    # Check for snow
    F.col("snow.1h") if "snow" in schema_fields else F.lit(None).alias("Snowfall (mm)"),
    F.col("weather").alias("Weather_Main"),
    F.col("Weather_Desc"),
]

# convert dt to datetime for storage too
# weather.main is things like rain, clouds, snow etc.

# # .toDF("IATA_number",)

raw_weather_df = raw_weather_df.select(*select_exprs).sort(F.asc("dt"))
raw_weather_df.show(n=1, truncate=False)